In [1]:
CONFIG_DATASET = 'VSL400'   # <-- SỬA: 'VSL400' hoặc 'VSL_GH'

STGCN_VERSION = 'new'
EPOCHS = 100
BATCH_SIZE = 64

# ---------------- AUTO CONFIG ----------------
if CONFIG_DATASET == 'VSL400':
    BASE = "/kaggle/input/datasets/vnnoname/vsl400-keypoints/vls400"
    TRAIN_DIR = f"{BASE}/train/keypoints"
    TEST_DIR  = f"{BASE}/test/keypoints"
    MAX_FRAMES = 80
elif CONFIG_DATASET == 'VSL_GH':
    BASE = "/kaggle/input/datasets/nguyenanfms/vsl-vietnamese-sign-language-v2/processed/processed/keypoints_splited"
    TRAIN_DIR = f"{BASE}/train"
    TEST_DIR  = f"{BASE}/test"
    MAX_FRAMES = 60

TAG = f"B2_{CONFIG_DATASET}_100ep_newstgcn"
CKPT_DIR = "/kaggle/working/checkpoints"
print(f"TAG: {TAG}")
print(f"CKPT_DIR: {CKPT_DIR}")

TAG: B2_VSL400_100ep_newstgcn
CKPT_DIR: /kaggle/working/checkpoints


In [2]:
#!pip install -q scikit-learn tqdm
import os, pickle, time, math, warnings, ctypes, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tqdm import tqdm
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [3]:
class CNNBiGRU(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, 3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.gru = nn.GRU(128, 128, bidirectional=True, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(256, num_classes)
        self.drop = nn.Dropout(0.3)
    def forward(self, x):
        x = x.permute(0,2,1)
        x = self.pool(F.relu(self.conv2(F.relu(self.conv1(x)))))
        x = x.permute(0,2,1)
        out, _ = self.gru(x)
        return self.fc(self.drop(out[:,-1,:]))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=128, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.embed = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout,
            batch_first=True, norm_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.embed(x); x = self.pos(x); x = self.transformer(x)
        x = self.norm(x); return self.fc(x.mean(dim=1))

def build_mediapipe_adjacency(num_joints=76):
    A = np.zeros((num_joints, num_joints), dtype=np.float32)
    pose_edges = [
        (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
        (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
        (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
        (11,23),(12,24),(23,24),
        (23,25),(25,27),(27,29),(29,31),(27,31),
        (24,26),(26,28),(28,30),(30,32),(28,32)]
    for i,j in pose_edges:
        if i<33 and j<33: A[i,j]=A[j,i]=1
    hand_edges = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
        (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
        (0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17)]
    for i,j in hand_edges:
        if 33+i<num_joints and 33+j<num_joints: A[33+i,33+j]=A[33+j,33+i]=1
        if 54+i<num_joints and 54+j<num_joints: A[54+i,54+j]=A[54+j,54+i]=1
    if num_joints>=55:
        A[15,33]=A[33,15]=1; A[16,54]=A[54,16]=1
    A += np.eye(num_joints)
    D = np.diag(1.0/np.sqrt(np.sum(A,axis=1)+1e-8))
    return (D@A@D).astype(np.float32)

class GraphConv(nn.Module):
    def __init__(self, in_c, out_c, adj):
        super().__init__()
        self.register_buffer('adj', torch.tensor(adj, dtype=torch.float32))
        self.weight = nn.Parameter(torch.empty(in_c, out_c))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, x):
        x = torch.einsum('bctv,co->botv', x, self.weight)
        return torch.einsum('botv,vw->botw', x, self.adj)

class STGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, adj, stride=1):
        super().__init__()
        self.gcn = GraphConv(in_c, out_c, adj)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.tcn = nn.Conv2d(out_c, out_c, kernel_size=(9,1), stride=(stride,1), padding=(4,0))
        self.bn2 = nn.BatchNorm2d(out_c)
        if in_c != out_c or stride != 1:
            self.residual = nn.Sequential(nn.Conv2d(in_c,out_c,1,stride=(stride,1)), nn.BatchNorm2d(out_c))
        else:
            self.residual = nn.Identity()
    def forward(self, x):
        res = self.residual(x)
        x = F.relu(self.bn1(self.gcn(x)))
        x = self.bn2(self.tcn(x))
        return F.relu(x + res)

class STGCN_New(nn.Module):
    def __init__(self, num_joints, num_classes, adj):
        super().__init__()
        self.data_bn = nn.BatchNorm1d(3*num_joints)
        self.blocks = nn.ModuleList([
            STGCNBlock(3,64,adj,stride=1), STGCNBlock(64,64,adj,stride=1),
            STGCNBlock(64,128,adj,stride=2), STGCNBlock(128,256,adj,stride=2)])
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        B, T, F_dim = x.shape
        V = F_dim//3
        x = x.view(B,T,V,3).permute(0,3,1,2).contiguous()
        x = x.view(B, 3*V, T)
        x = self.data_bn(x)
        x = x.view(B,3,V,T).permute(0,1,3,2)
        for b in self.blocks: x = b(x)
        return self.fc(x.mean(dim=(2,3)))

class TCN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, 3, padding=2, dilation=2)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, 3, padding=4, dilation=4)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 256, 3, padding=8, dilation=8)
        self.bn3 = nn.BatchNorm1d(256)
        self.fc = nn.Linear(256, num_classes)
        self.drop = nn.Dropout(0.3)
    def forward(self, x):
        x = x.permute(0,2,1)
        x = self.drop(F.relu(self.bn1(self.conv1(x))))
        x = self.drop(F.relu(self.bn2(self.conv2(x))))
        x = self.drop(F.relu(self.bn3(self.conv3(x))))
        return self.fc(x.mean(dim=2))

In [4]:
class VSLDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i].contiguous(), self.y[i]

def load_split(base_dir, gloss_to_id, max_frames):
    X_list, y_list = [], []
    gloss_dirs = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])
    for gloss in tqdm(gloss_dirs, desc="Loading"):
        if gloss not in gloss_to_id: continue
        label = gloss_to_id[gloss]
        gp = os.path.join(base_dir, gloss)
        for fname in os.listdir(gp):
            if not fname.endswith('.npy'): continue
            kp = np.load(os.path.join(gp, fname)).reshape(-1, 76*3)
            T = kp.shape[0]
            if T > max_frames: kp = kp[:max_frames]
            elif T < max_frames:
                pad = np.zeros((max_frames-T, kp.shape[1]), dtype=kp.dtype)
                kp = np.concatenate([kp, pad], axis=0)
            X_list.append(kp); y_list.append(label)
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int64)

train_glosses = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
gloss_to_id = {g: i for i, g in enumerate(train_glosses)}
print(f"Classes: {len(gloss_to_id)}")

print("Loading train...")
X_train_full, y_train_full = load_split(TRAIN_DIR, gloss_to_id, MAX_FRAMES)
print("Loading test...")
X_test, y_test = load_split(TEST_DIR, gloss_to_id, MAX_FRAMES)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1, random_state=42, stratify=y_train_full)

input_dim = X_train.shape[2]
num_classes = len(gloss_to_id)
num_joints = input_dim // 3
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

train_loader = DataLoader(VSLDataset(X_train, y_train), BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(VSLDataset(X_val, y_val), BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(VSLDataset(X_test, y_test), BATCH_SIZE, shuffle=False, num_workers=2)

Classes: 400
Loading train...


Loading: 100%|██████████| 400/400 [01:16<00:00,  5.26it/s]


Loading test...


Loading: 100%|██████████| 400/400 [00:17<00:00, 23.43it/s]


Train: (18214, 80, 228), Val: (2024, 80, 228), Test: (4514, 80, 228)


In [5]:
def train_one_epoch(model, loader, opt, crit):
    model.train(); tl=tc=tt=0
    for X, y in tqdm(loader, desc='Train', leave=False):
        X, y = X.to(device), y.to(device)
        opt.zero_grad()
        out = model(X); loss = crit(out, y)
        loss.backward(); opt.step()
        tl += loss.item(); tc += (out.argmax(1)==y).sum().item(); tt += y.size(0)
    return tl/len(loader), tc/tt

def evaluate(model, loader, crit):
    model.eval(); tl=tc=tt=0; preds=[]; labels=[]
    with torch.no_grad():
        for X, y in tqdm(loader, desc='Eval', leave=False):
            X, y = X.to(device), y.to(device)
            out = model(X); tl += crit(out, y).item()
            p = out.argmax(1); tc += (p==y).sum().item(); tt += y.size(0)
            preds.extend(p.cpu().numpy()); labels.extend(y.cpu().numpy())
    return tl/len(loader), tc/tt, preds, labels

def measure_inference(model, loader, warmup=5, measure=20):
    model.eval(); times=[]
    with torch.no_grad():
        for i, (X, _) in enumerate(loader):
            if i >= warmup+measure: break
            X = X.to(device)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t0=time.time(); _ = model(X)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t1=time.time()
            if i >= warmup: times.append(t1-t0)
    return np.mean(times)

def _clean_state_dict(model):
    state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    return {k.replace('module.', ''): v.detach().cpu() for k, v in state.items()}

def train_model(model, name, lr):
    opt = optim.Adam(model.parameters(), lr=lr); crit = nn.CrossEntropyLoss()
    best = -1.0; best_epoch = 0
    hist = {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[]}
    os.makedirs(CKPT_DIR, exist_ok=True)
    ckpt_path = os.path.join(CKPT_DIR, f"{TAG}_{name}_best.pth")
    t0 = time.time()
    for ep in range(1, EPOCHS+1):
        trl, tra = train_one_epoch(model, train_loader, opt, crit)
        vll, vla, _, _ = evaluate(model, val_loader, crit)
        hist['train_loss'].append(trl); hist['train_acc'].append(tra)
        hist['val_loss'].append(vll); hist['val_acc'].append(vla)
        print(f"Epoch {ep}/{EPOCHS} | tr_loss={trl:.4f} tr_acc={tra:.4f} | val_loss={vll:.4f} val_acc={vla:.4f}")
        if vla > best:
            best = vla; best_epoch = ep
            torch.save({
                'model_name': name,
                'tag': TAG,
                'epoch': ep,
                'val_acc': float(vla),
                'state_dict': _clean_state_dict(model),
            }, ckpt_path)
            print(f"Saved best checkpoint: {ckpt_path} (epoch={ep}, val_acc={vla:.4f})")
    train_time = time.time() - t0
    try:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt['state_dict']
    if hasattr(model, 'module'):
        model.module.load_state_dict(state_dict)
    else:
        model.load_state_dict(state_dict)
    inf_batch = measure_inference(model, test_loader)
    inf_ms = inf_batch / BATCH_SIZE * 1000
    _, test_acc, preds, labels = evaluate(model, test_loader, crit)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    pw, rw, f1w, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    return {'accuracy':test_acc,'macro_f1':f1,'weighted_f1':f1w,'macro_precision':p,'macro_recall':r,
            'train_time_s':train_time,'inference_ms':inf_ms,
            'best_val_acc':best,'best_epoch':best_epoch,'checkpoint':ckpt_path}, hist, preds, labels

In [ ]:
ADJ = build_mediapipe_adjacency(num_joints=num_joints)

LR_CONFIG = {'CNN-BiGRU':1e-3, 'Transformer':1e-4, 'ST-GCN':1e-3, 'TCN':1e-3}
models = {
    # 'CNN-BiGRU':   CNNBiGRU(input_dim, num_classes),
    # 'Transformer': TransformerModel(input_dim, num_classes),
    'ST-GCN':      STGCN_New(num_joints, num_classes, ADJ),
    'TCN':         TCN(input_dim, num_classes),
}

all_results, all_hist, all_preds, all_labels = {}, {}, {}, {}
for name, model in models.items():
    print(f"\n{'='*60}\nTraining {name} on {TAG}\n{'='*60}")
    model = model.to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Params: {n_params:,}")
    m, h, p, l = train_model(model, name, LR_CONFIG[name])
    m['n_params'] = n_params
    all_results[name] = m; all_hist[name] = h; all_preds[name] = p; all_labels[name] = l
    print(f"✅ {name}: Acc={m['accuracy']:.4f}, F1={m['macro_f1']:.4f}, Inf={m['inference_ms']:.2f}ms")
    print(f"Checkpoint: {m['checkpoint']}")
    del model; gc.collect(); torch.cuda.empty_cache()
    ctypes.CDLL("libc.so.6").malloc_trim(0)

# ---------------- SAVE ----------------
df = pd.DataFrame(all_results).T
print(f"\n=== FINAL RESULTS - {TAG} ===")
print(df.round(4))
df.to_csv(f'{TAG}.csv')
with open(f'{TAG}.pkl', 'wb') as f:
    pickle.dump({'tag':TAG,'config':{'dataset':CONFIG_DATASET,'epochs':EPOCHS,'stgcn':STGCN_VERSION},
                 'results':all_results,'history':all_hist,'preds':all_preds,
                 'labels':all_labels,'gloss_names':train_glosses}, f)
print(f"\n✅ Saved: {TAG}.csv và {TAG}.pkl")


Training Transformer on B2_VSL400_100ep_newstgcn
Params: 2,453,264


Epoch 1/100 | tr_loss=5.8355 tr_acc=0.0087 | val_loss=5.3493 val_acc=0.0346
Saved best checkpoint: /kaggle/working/checkpoints/B2_VSL400_100ep_newstgcn_Transformer_best.pth (epoch=1, val_acc=0.0346)


Epoch 2/100 | tr_loss=5.0200 tr_acc=0.0610 | val_loss=4.6013 val_acc=0.1206
Saved best checkpoint: /kaggle/working/checkpoints/B2_VSL400_100ep_newstgcn_Transformer_best.pth (epoch=2, val_acc=0.1206)


Epoch 3/100 | tr_loss=4.3054 tr_acc=0.1769 | val_loss=3.8677 val_acc=0.2792
Saved best checkpoint: /kaggle/working/checkpoints/B2_VSL400_100ep_newstgcn_Transformer_best.pth (epoch=3, val_acc=0.2792)


Epoch 4/100 | tr_loss=3.6147 tr_acc=0.3336 | val_loss=3.2427 val_acc=0.3948
Saved best checkpoint: /kaggle/working/checkpoints/B2_VSL400_100ep_newstgcn_Transformer_best.pth (epoch=4, val_acc=0.3948)


...

Epoch 56/100 | tr_loss=0.0647 tr_acc=0.9820 | val_loss=0.8049 val_acc=0.8503


Epoch 57/100 | tr_loss=0.0403 tr_acc=0.9896 | val_loss=0.8740 val_acc=0.8439


Epoch 58/100 | tr_loss=0.0564 tr_acc=0.9847 | val_loss=0.9509 val_acc=0.8226


Epoch 59/100 | tr_loss=0.0677 tr_acc=0.9800 | val_loss=0.8613 val_acc=0.8444


Train:  97%|█████████▋| 277/285 [01:08<00:01,  4.04it/s]

In [1]:
import logging
import os
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
logger = logging.getLogger(__name__)

working_dir = Path("/kaggle/working")

if not working_dir.is_dir():
    logger.error("Directory does not exist: %s", working_dir)
else:
    for root, dirs, files in os.walk(working_dir):
        rel_root = Path(root).relative_to(working_dir)
        for name in sorted(dirs):
            path = rel_root / name
            logger.info("[dir]  %s", path if str(path) != "." else name)
        for name in sorted(files):
            path = (rel_root / name) if str(rel_root) != "." else Path(name)
            size = (Path(root) / name).stat().st_size
            logger.info("[file] %s  (%s bytes)", path, size)

2026-09-30 10:22:08,749 INFO [dir]  .virtual_documents
2026-09-30 10:22:08,750 INFO [dir]  checkpoints
2026-09-30 10:22:08,751 INFO [file] checkpoints/B2_VSL400_100ep_newstgcn_CNN-BiGRU_best.pth  (1483833 bytes)
2026-09-30 10:22:08,752 INFO [file] checkpoints/B2_VSL400_100ep_newstgcn_ST-GCN_best.pth  (4152345 bytes)
2026-09-30 10:22:08,752 INFO [file] checkpoints/B2_VSL400_100ep_newstgcn_Transformer_best.pth  (12392792 bytes)
